In [ ]:
import math
import copy
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from itertools import count
from importnb import Notebook
from typing import Any, List, Tuple, Dict

with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv
    from LabUserRequest import UserRequestEvents
    from LabPrefetchScheduler import PrefetchScheduler

import os
import sys

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

import Common.config as config
import Common.datatypes as datatypes
import Common.utils as utils

import importlib

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(utils)

TransitionTuple = datatypes.TransitionTuple

In [ ]:
class EnvWrapper(gym.Env):
    """
    Simple wrapper that delegates all calls to an inner env.
    Subclass this to create your own wrappers.
    """

    metadata = {"render.modes": []}

    def __init__(
        self, 
        cfg: config.Config,
        n: int,
        m: int,
        n_layers: int,
        lam: float,
        theta: float,
        users_env: None,
        du_caches: None,
        mec_cache: None,
        latency_model: None,
        prefetch_fn: None,
        reward_fn: None,
        max_steps: int = 10000,
        *,
        step_duration_s=1.0,
        debugger=None
    ):
        super().__init__()
        self.cfg = cfg

        self.step_count = 0
        self.step_duration_s = step_duration_s
        self.max_steps = max_steps

        self.n = n  # number of tiles per row/column
        self.m = m  # number of tiles per row/column
        self.n_layers = n_layers  # number of layers (base + enhancement)
        
        self.gain_if_prefetched = 1.0
        self.loss_if_not_prefetched = -1.0

        self.theta = theta
        self.lam = lam

        self.users_env = users_env
        self.du_caches = du_caches
        self.mec_cache = mec_cache
        self.latency_model = latency_model

        self.prefetch_fn = prefetch_fn or (lambda cache, action: cache.drl_prefetching(action))
        self.reward_fn = reward_fn or (lambda info: info.get('reward_per_user', {}).get(info.get('current_user', -1), 0.0))

        # History holders
        self.users_reward = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.users_psnr = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.total_gop_requests_per_user = {
            u: 0 for u in range(self.users_env.n_users)
        }
        
        self.scheduler = PrefetchScheduler(
            R_M_D=self.latency_model.R_M_D,
            R_C_M=self.latency_model.R_C_M,
            U=self.latency_model.max_U,
            step_duration_s=self.step_duration_s
        )

        self.debugger = debugger
        
    # ─────────────────────────────────────────────────────────────────────────
    # Internal Helpers
    # ─────────────────────────────────────────────────────────────────────────
    def _make_ready_bitmaps(self, du_planned, mec_planned):
        du_ready = None
        mec_ready = None

        if du_planned:
            du_ready = []
            for du_idx, bm in enumerate(du_planned):
                cache_key = f"DU:{du_idx}"
                du_ready.append(self.scheduler.materialize_ready_bitmap(cache_key, bm))

        if mec_planned is not None:
            mec_ready = self.scheduler.materialize_ready_bitmap("MEC", mec_planned)

        return du_ready, mec_ready

    def _missing_items(self, req: Dict[str, Any]) -> list[int]:
        missing = 4 * [0]

        video_cache_index = self.mec_cache.policy.video_idx
        tile_cache_index = self.mec_cache.policy.tile_idx
        
        video = req["video"]
        viewport = req["viewport"]
        
        if video not in video_cache_index:
            missing = [1, 1, 1, 1]  # All enhancement layers missing
        else:
            video_idx = video_cache_index.index(video)
            cached_tiles = tile_cache_index[video_idx]
                        
            for idx, tile in enumerate(viewport):
                if tile not in cached_tiles:
                    missing[idx] = 1  # Enhancement layer missing

        return missing

    def _create_transition(self, state, action_idx) -> TransitionTuple:
        """
        Create frozen DRL transition snapshot.
        Deep copy avoids mutation by stateful feature updates.
        """
        return TransitionTuple(
            state=copy.deepcopy(state),
            action=action_idx,
            reward=0.0,
            next_state=None
        )

    def _process_prefetch_actions(
        self,
        req,
        agent,
        net_adapter,
        du_plan,
        mec_plan
    ) -> Tuple[List[TransitionTuple], Any, Any]:

        user = req["u"]
        video = req["video"]
        viewport = req["viewport"]

        transitions = [None] * 5
        missing = self._missing_items(req)

        for idx, is_missing in enumerate(missing):
            if not is_missing:
                continue

            # Build state representation
            state = net_adapter.build_observation(
                req, video, viewport[idx - 1] if idx > 0 else None
            )

            video_cache_idx = self.mec_cache.get_video_cache_idx(video)
            if idx > 0 and video_cache_idx == -1:
                break

            # DRL action selection
            action_idx, _ = agent.select_action(state, idx, video_cache_idx)

            action = {
                "user": user,
                "video": video,
                "tiles": [] if idx == 0 else [viewport[idx - 1]],
                "base_req_init": (idx == 0),
                "action_idx": action_idx
            }

            # Apply to MEC cache
            mec_plan = self.prefetch_fn(self.mec_cache, action)

            # Create immutable transition snapshot
            transitions[idx] = self._create_transition(state, action_idx)

        return transitions, du_plan, mec_plan

    def _compute_cache_hits(self, req):
        video = req["video"]
        viewport = req["viewport"]

        video_cache_index = self.mec_cache.policy.video_idx
        tile_cache_index = self.mec_cache.policy.tile_idx

        base_hit = 1 if video in video_cache_index else 0
        
        video_idx = self.mec_cache.get_video_cache_idx(video)

        if video_idx != -1:
            enh_hit = [
                1 if tile in tile_cache_index[video_idx] else 0 for tile in viewport
            ]
        else:
            enh_hit = [0, 0, 0, 0]
    
        return dict(
            base_layer_hits=12 * base_hit,
            enh_layer_hits=sum(enh_hit),
            base_layer_misses=12 * (1 - base_hit),
            enh_layer_misses=4 - sum(enh_hit)
        )
          
    # ─────────────────────────────────────────────────────────────────────────
    # Gym Environment API
    # ─────────────────────────────────────────────────────────────────────────
    def step(self, agent=None, net_adapter=None, req=None, cfg=None):
        info = {}

        du_plan = self.du_caches if len(self.du_caches) > 0 else None
        mec_plan = self.mec_cache.get_cache_bitmap() if self.mec_cache else None

        # ------------------------------------------------------------
        # 1. Handle each user's missing tiles with DRL decisions
        # ------------------------------------------------------------
        transitions, du_plan, mec_plan = self._process_prefetch_actions(
            req, agent, net_adapter, du_plan, mec_plan
        )

        nxt_req = self.users_env.get_next_request(du_plan, mec_plan)

        reward_val = net_adapter.features.compute_reward()
        if nxt_req is None:
            nxt_req = req

        for idx, trans in enumerate(transitions):
            if trans is None:
                continue

            nxt_state = net_adapter.build_observation(
                nxt_req, nxt_req["video"], nxt_req["viewport"][idx - 1] if idx > 0 else None
            )

            trans.next_state = nxt_state
            transitions[idx].reward = reward_val

            agent.remember(
                transitions[idx].state,
                transitions[idx].action,
                transitions[idx].reward,
                transitions[idx].next_state,
                done=self.users_env.all_users_done()
            )
        
            agent.train_step()

        # -------------------------------------------------------
        # 3. Remaining users requests
        # -------------------------------------------------------
        info["user_request"] = nxt_req

        # -------------------------------------------------------
        # 4. Cache stats (HIT / MISS)
        # -------------------------------------------------------
        info.update(self._compute_cache_hits(nxt_req))

        # -------------------------------------------------------
        # 5. Compute final per-user reward
        # -------------------------------------------------------
        reward = reward_val

        # -------------------------------------------------------
        # 6. Advance simulation time (1 step)
        # -------------------------------------------------------
        done = self.users_env.all_users_done() or (self.step_count >= self.max_steps - 1)
        self.step_count += 1

        return {}, reward, done, info


    # ---------------------------------------------------------
    # REWARD FUNCTION
    # ---------------------------------------------------------
    def compute_reward(self, psnr_per_user):
        reward_per_user = {}
        for u, psnr in psnr_per_user.items():    
            reward_per_user[u] = psnr

        return reward_per_user

    # ---------------------------------------------------------
    #  SAMPLE ACTION (for testing)
    # ---------------------------------------------------------
    def sample_action(self):
        tiles = np.zeros(self.n * self.m, dtype=int)
        c = self.n // 2
        if self.n % 2 == 1:
            center_idx = c * self.n + c
            tiles[center_idx] = 1
        else:
            centers = [(c-1, c-1), (c-1, c), (c, c-1), (c, c)]
            for x, y in centers:
                tiles[y * self.n + x] = 1

        return {
            'video': np.random.randint(0, self.users_env.n_videos),
            'gop': np.random.randint(0, self.users_env.n_gops),
            'tiles': tiles.tolist()
        }, c * self.n + c

    # ---------------------------------------------------------
    # WARM-UP PHASE
    # ---------------------------------------------------------
    def warmup_phase(self, net_adapter, num_steps=1000):
        """Warm-up phase to initialize cache with some videos."""

        base_idx_counter = 1
        enh_idx_counter = 1
        
        for _ in range(num_steps):  # Arbitrary number of warm-up steps
            req = self.users_env.get_next_request(None, None)
            if req is None:
                break

            video = req["video"]
            action = {
                "video": video,
                "tiles": [],
                "base_req_init": True,
                "action_idx": base_idx_counter # Dummy action index for warm-up
            }
            self.prefetch_fn(self.mec_cache, action)
            
            viewport = req["viewport"]
            for idx in range(4):
                tile_action = {
                    "video": video,
                    "tiles": [viewport[idx]],
                    "base_req_init": False,
                    "action_idx": self.cfg.cache_size + base_idx_counter * 4 + 1 + enh_idx_counter
                }
                self.prefetch_fn(self.mec_cache, tile_action)
                
                enh_idx_counter = (enh_idx_counter + 1) % self.cfg.viewport
            
            base_idx_counter = (base_idx_counter + 1) % self.cfg.cache_size
            net_adapter.features.update_history(video, viewport)

            print(f"Warm-up step {base_idx_counter} {enh_idx_counter} | Video {video} prefetched with tiles {viewport}")

    # ---------------------------------------------------------
    # RESET
    # ---------------------------------------------------------
    def reset(self, **kwargs):
        self.step_count = 0
        
        self.users_reward = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.users_psnr = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.total_gop_requests_per_user = {
            u: 0 for u in range(self.users_env.n_users)
        }

        _, info_users = self.users_env.reset(**kwargs)

        info_cache_mec = self.mec_cache.reset(**kwargs)[1] if self.mec_cache else {}

        req = self.users_env.get_next_request(None, None)

        info_cache = {
            **info_cache_mec,
            **info_users,
            "user_request": req
        }
        
        # Reset scheduler's time and availability
        self.scheduler.now_s = 0.0
        self.scheduler.availability = {}

        return None, info_cache

In [3]:
def getTiles(step, user, users_viewport_tiles, n) -> np.ndarray:
    mask = np.zeros(n * n, dtype=int)    
    for tx, ty in users_viewport_tiles[user][step]:
        if 0 <= tx < n and 0 <= ty < n:
            mask[ty * n + tx] = 1
    
    return mask

In [4]:
if __name__ == "__main__":
    n_episodes = 100
    n_nodes = 3
    n_users = 100
    step_size = 5.0
    alpha = 1.0
    n_gops = 60
    n_layers = 2
    n = 4
    max_capacity = 5000e6  # 5000 MB
    n_videos = 1000

    #### CPT parameters ####
    lam = 3.7183
    theta = 0.5

    users_env = UserRequestEvents(
        n_nodes=n_nodes,
        n_users=n_users,
        step_size=step_size,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n*n,
        n=n,
        alpha=alpha,
        users_viewport_tiles=None,
        requested_videos=None
    )

    # du_caches = [
    #     CacheEngineEnv(
    #         n_tiles=n*n,
    #         n_videos=n_videos,
    #         cache_capacity=max_capacity
    #     ) for _ in range(n_nodes)  # Number of DUs = n_nodes
    # ]
    du_caches = []

    mec_cache = CacheEngineEnv(
        n_tiles=n*n,
        n_videos=n_videos,
        cache_capacity=max_capacity
    )

    # Create latency model (replace numbers with your real config)
    P = n_nodes; max_U = n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,       # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9,     # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 40e6, dtype=float),    # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        lam=lam,
        theta=theta
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {n_users}\n"
        f"Cache capacity: {max_capacity}MB\n"
        f"Video matrix: {n_videos}x{n_layers}x{n*n}\n"
        f"==============================\n"
    )

    _, info = env.reset()
    viewport_tiles = info['viewport_tiles']
    requested_videos = info['requested_videos']

    done = False
    results = []
    total_reward = 0.0

    for step in count():
        actions = [
            {
                'user': user,
                'video': requested_videos[user],
                'tiles': getTiles(step % n_gops, user, viewport_tiles, n),
                'gop': step % n_gops
            } for user in range(n_users)
        ]

        obs, reward, done, info = env.step(actions)

        total_reward += float(reward)

        results.append({
            "step": step,
            "total_reward": total_reward,
            "cache_hits": info["base_layer_cache_hits"] + info["enh_layer_cache_hits"],
            "cache_misses": info["base_layer_cache_misses"] + info["enh_layer_cache_misses"],
            "info": info
        })

        print(
            f"Step {step} - "
            f"Reward: {reward:.4f} - "
            f"Total Reward: {total_reward:.4f} - "
            f"Cache Hits: {info['base_layer_cache_hits'] + info['enh_layer_cache_hits']} - "
            f"Cache Misses: {info['base_layer_cache_misses'] + info['enh_layer_cache_misses']}\n"
            f"Cache Utilization: {info['cache_utilization']:.2%} - "
            f"Items in Cache: {info['cache_num_items']} - "
            f"Final Capacity: {info['cache_current_capacity']/1e6:.2f}/{info['cache_total_capacity']/1e6:.1f} MB"
        )
        
        if done: 
            break

TypeError: UserRequestEvents.__init__() got an unexpected keyword argument 'alpha'

In [ ]:
if __name__ == "__main__":
    steps = range(1, len(results) + 1)
    cache_hits_series = [r["cache_hits"] for r in results]
    cache_misses_series = [r["cache_misses"] for r in results]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=True)

    # Rewards with moving average
    axes[0].plot(steps, [r["total_reward"] for r in results], label="Step reward", alpha=0.7, color="blue")
    w = max(1, min(20, len(results) // 10))
    if w > 1:
        ma = [sum([r["total_reward"] for r in results][i - w:i]) / w for i in range(w, len(results) + 1)]
        axes[0].plot(range(w, len(results) + 1), ma, label=f"Moving avg (w={w})", color="orange")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Total reward")
    axes[0].set_title("Training Rewards")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Cache hits
    axes[1].plot(steps, cache_hits_series, label="Cache hits", color="green", alpha=0.8)
    axes[1].set_title("Cache Hits per Step")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Hits")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    # Cache misses
    axes[2].plot(steps, cache_misses_series, label="Cache misses", color="red", alpha=0.8)
    axes[2].set_title("Cache Misses per Step")
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("Misses")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == "__main__":
    n_episodes = 100
    n_nodes = 3
    n_users = 100
    step_size = 5.0
    alpha = 1.0
    n_gops = 60
    n_layers = 2
    n = 4
    max_capacity = 5000e6  # 5000 MB
    n_videos = 1000

    #### CPT parameters ####
    lam = 3.7183
    theta = 0.5

    users_env = UserTileRequestEvents(
        n_users=n_users,
        step_size=step_size,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n*n,
        n=n,
        alpha=alpha,
        users_viewport_tiles=None,
        requested_videos=None
    )

    du_caches = []

    mec_cache = CacheEngineEnv(
        n_tiles=n*n,
        n_videos=n_videos,
        cache_capacity=max_capacity
    )

    # Create latency model (replace numbers with your real config)
    P = n_nodes; max_U = n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,       # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9,     # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 40e6, dtype=float),    # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        lam=lam,
        theta=theta
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {n_users}\n"
        f"Cache capacity: {max_capacity}MB\n"
        f"Video matrix: {n_videos}x{n_layers}x{n*n}\n"
        f"==============================\n"
    )

    _, info = env.reset()
    viewport_tiles = info['viewport_tiles']
    requested_videos = info['requested_videos']

    done = False
    results = []
    total_reward = 0.0

    for step in count():
        actions = [
            {
                'user': user,
                'video': requested_videos[user],
                'tiles': getTiles(step % n_gops, user, viewport_tiles, n),
                'gop': step % n_gops
            } for user in range(n_users)
        ]

        obs, reward, done, info = env.step(actions)

        total_reward += float(reward)

        results.append({
            "step": step,
            "total_reward": total_reward,
            "cache_hits": info["base_layer_cache_hits"] + info["enh_layer_cache_hits"],
            "cache_misses": info["base_layer_cache_misses"] + info["enh_layer_cache_misses"],
            "info": info
        })

        print(
            f"Step {step} - "
            f"Reward: {reward:.4f} - "
            f"Total Reward: {total_reward:.4f} - "
            f"Cache Hits: {info['base_layer_cache_hits'] + info['enh_layer_cache_hits']} - "
            f"Cache Misses: {info['base_layer_cache_misses'] + info['enh_layer_cache_misses']}\n"
            f"Cache Utilization: {info['cache_utilization']:.2%} - "
            f"Items in Cache: {info['cache_num_items']} - "
            f"Final Capacity: {info['cache_current_capacity']/1e6:.2f}/{info['cache_total_capacity']/1e6:.1f} MB"
        )
        
        if done: 
            break